In [35]:
import pandas as pd
from pathlib import Path
import re

Definição do project root e pastas

In [36]:
PROJECT_ROOT = Path.cwd().parent

# Pastas de entrada e saída
DATASET_VISEM = PROJECT_ROOT / "dataset_visem" / "Train"
DATA_RAW = PROJECT_ROOT / "data" / "raw"

# Garante que a pasta de saída exista
DATA_RAW.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset exists:", DATASET_VISEM.exists())
print("Raw data folder:", DATA_RAW)


Project root: c:\Users\ccana\Documents\Doutorado\VISEMTracking
Dataset exists: True
Raw data folder: c:\Users\ccana\Documents\Doutorado\VISEMTracking\data\raw


In [37]:
users = [11,12,13,14,15,19,21,22,23,24,29,30,35,36,38,47,52,54,60,82]  # IDs dos usuários
all_users_frames = []

Leitura e consolidação dos artigos txt do VISEM

In [38]:
for user in users:
    print(f"\n=== User {user} ===")

    data_folder = DATASET_VISEM / str(user) / "labels_ftid"
    print(f"Pasta de dados: {data_folder}")

    if not data_folder.exists():
        print(f"AVISO: Pasta não existe: {data_folder}")
        continue

    txt_files = sorted(data_folder.glob("*.txt"))
    print(f"Quantidade de arquivos .txt encontrados: {len(txt_files)}")

    if not txt_files:
        print(f"AVISO: Nenhum arquivo .txt encontrado para o usuário {user}")
        continue

    frames_list = []

    for file_path in txt_files:
        file_name = file_path.name

        # === Extração do número do frame ===
        match = re.search(r"_frame_(\d+)_with_ftid\.txt$", file_name)

        if not match:
            print(f"AVISO: Nome de arquivo fora do padrão: {file_name}")
            continue

        frame_number = int(match.group(1))

        try:
            df = pd.read_csv(
                file_path,
                sep=r"\s+",
                header=None,
                names=["trajectory_id", "ftid", "x", "y", "w", "h"]
            )
        except Exception as e:
            print(f"Erro ao ler {file_name}: {e}")
            continue

        df["timestamp"] = frame_number
        df["user_id"] = str(user)

        frames_list.append(df)

    if not frames_list:
        print(f"AVISO: Nenhum DataFrame válido criado para o usuário {user}")
        continue

    trajectories_df = pd.concat(frames_list, ignore_index=True)

    # Reorganiza as colunas
    trajectories_df = trajectories_df[
        ["user_id", "timestamp", "trajectory_id", "ftid", "x", "y", "w", "h"]
    ]

    print(f"DataFrame do usuário {user} criado com sucesso")
    print(f"Total de linhas: {len(trajectories_df)}")

    all_users_frames.append(trajectories_df)


=== User 11 ===
Pasta de dados: c:\Users\ccana\Documents\Doutorado\VISEMTracking\dataset_visem\Train\11\labels_ftid
Quantidade de arquivos .txt encontrados: 1470
DataFrame do usuário 11 criado com sucesso
Total de linhas: 56568

=== User 12 ===
Pasta de dados: c:\Users\ccana\Documents\Doutorado\VISEMTracking\dataset_visem\Train\12\labels_ftid
Quantidade de arquivos .txt encontrados: 1470
DataFrame do usuário 12 criado com sucesso
Total de linhas: 39357

=== User 13 ===
Pasta de dados: c:\Users\ccana\Documents\Doutorado\VISEMTracking\dataset_visem\Train\13\labels_ftid
Quantidade de arquivos .txt encontrados: 1470
DataFrame do usuário 13 criado com sucesso
Total de linhas: 62646

=== User 14 ===
Pasta de dados: c:\Users\ccana\Documents\Doutorado\VISEMTracking\dataset_visem\Train\14\labels_ftid
Quantidade de arquivos .txt encontrados: 1470
DataFrame do usuário 14 criado com sucesso
Total de linhas: 5750

=== User 15 ===
Pasta de dados: c:\Users\ccana\Documents\Doutorado\VISEMTracking\dat

In [39]:
if all_users_frames:
    all_df = pd.concat(all_users_frames, ignore_index=True)

    output_file = DATA_RAW / "visem_all_trajectory.csv"
    all_df.to_csv(output_file, index=False)

    print(f"\nFinal CSV saved at: {output_file}")
    print(f"Total rows (all users): {len(all_df)}")
else:
    print("\nNo valid user data was generated.")



Final CSV saved at: c:\Users\ccana\Documents\Doutorado\VISEMTracking\data\raw\visem_all_trajectory.csv
Total rows (all users): 656335
